# Inferencia D20 V1

Carrega o modelo treinado, detecta a face visivel e calcula a face oposta com a regra 21 - n.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone

import cv2

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ProjectConfig
from src.data_io import list_image_paths
from src.cv_pipeline import image_to_feature
from src.modeling import load_model_bundle
from src.ocr_ensemble import read_number_ensemble
from src.d20_rules import opposite_face
from src.schema import write_results_json

cfg = ProjectConfig()
bundle = load_model_bundle(PROJECT_ROOT / cfg.model_path)
classifier = bundle['classifier']
model_version = bundle.get('metadata', {}).get('model_version', cfg.model_version)

print('Modelo carregado com sucesso')
print('Versao:', model_version)

In [ ]:
image_paths = list_image_paths(PROJECT_ROOT / cfg.input_dir)
print(f'Imagens para inferencia: {len(image_paths)}')
if not image_paths:
    raise RuntimeError('Nenhuma imagem encontrada para inferencia em input/d20.')

In [ ]:
results = []

for image_path in image_paths:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        continue

    feature, meta, number_roi = image_to_feature(image)
    pred_label, pred_conf = classifier.predict(feature)

    ocr_label, ocr_conf, ocr_source = read_number_ensemble(number_roi)

    face_inferida = int(pred_label)
    confianca = float(pred_conf)

    if ocr_label is not None and ocr_conf > (pred_conf + 0.10):
        face_inferida = int(ocr_label)
        confianca = float(ocr_conf)

    face_oposta = opposite_face(face_inferida)

    results.append({
        'imagem': image_path.name,
        'face_inferida': face_inferida,
        'face_oposta': face_oposta,
        'confianca': round(confianca, 4),
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'versao_modelo': model_version,
        '_debug_origem': 'ocr' if (ocr_label is not None and ocr_conf > (pred_conf + 0.10)) else 'modelo',
        '_debug_bbox': meta.get('bbox'),
        '_debug_ocr_source': ocr_source,
    })

print(f'Resultados calculados: {len(results)}')

In [ ]:
# Remove campos de debug antes de persistir o JSON V1 enxuto
public_results = []
for item in results:
    public_results.append({
        'imagem': item['imagem'],
        'face_inferida': item['face_inferida'],
        'face_oposta': item['face_oposta'],
        'confianca': item['confianca'],
        'timestamp': item['timestamp'],
        'versao_modelo': item['versao_modelo'],
    })

write_results_json(public_results, PROJECT_ROOT / cfg.results_path)
print(f'Arquivo salvo em: {PROJECT_ROOT / cfg.results_path}')

for item in public_results[:5]:
    print(item)

In [ ]:
# Exemplo de regra esperada: se face_inferida = 2, face_oposta = 19
example = next((r for r in public_results if r['face_inferida'] == 2), None)
if example:
    print('Exemplo encontrado:', example['imagem'])
    print('Calculo aplicado: 21 - 2 =', example['face_oposta'])
else:
    print('Nenhum exemplo com face 2 encontrado no lote atual.')